# 8. STAC Browser Against a Secured API

Chapters 6 and 7 exercised the auth layer with `httpx` — explicit headers, explicit
assertions. That is the right way to *verify* a policy, but it is not how anyone
experiences a catalog.

This chapter connects [STAC Browser](https://github.com/radiantearth/stac-browser) to the
same proxied API, logs in as different users, and watches the catalog change shape in a
real UI. Nothing about the policy changes — only the client.

<div class="alert alert-block alert-warning">
<b>Note:</b> This chapter needs the local docker-compose stack. It will not run against the
hosted workshop deployment.
</div>

## 8.1 How the client learns it can authenticate

A STAC API that requires auth needs to tell clients *how* to authenticate. That is what the
[Authentication Extension](https://github.com/stac-extensions/authentication) is for: it
advertises the available auth schemes in the catalog's own JSON.

stac-auth-proxy adds this for you — `ENABLE_AUTHENTICATION_EXTENSION` defaults to `true`,
which is why we never set it in `docker-compose.yml`. Let's look at what the proxy is
publishing.

In [ ]:
import json
import httpx

from stac_auth import auth_headers, get_mock_oidc_token, require_local_auth_stack

stac_api_endpoint, mock_oidc_endpoint = require_local_auth_stack()

landing = httpx.get(stac_api_endpoint, timeout=10).json()

schemes = landing.get("auth:schemes", {})
print(json.dumps(schemes, indent=2))

assert schemes, "Expected the proxy to advertise auth:schemes"

The scheme object describes an OpenID Connect provider and where to discover it. STAC
Browser's `authConfig` option takes *this same shape* — it is the Authentication Extension's
scheme object, so what the proxy advertises and what the browser is configured with line up
by design.

<div class="alert alert-block alert-info">
STAC Browser does not yet read this automatically; you still configure it. But the shared
shape means you are copying a value, not translating between two formats.
</div>

## 8.2 Configuring STAC Browser

The `stac-browser` service in `docker-compose.yml`:

```yaml
environment:
  SB_catalogUrl: "http://localhost:8084"
  SB_authConfig: >-
    {"type":"openIdConnect",
    "openIdConnectUrl":"http://localhost:8085/.well-known/openid-configuration",
    "oidcConfig":{"client_id":"stac-browser","automaticSilentRenew":false}}
```

Four important details:

- **`SB_catalogUrl` points at the proxy (`:8084`), not the upstream API (`:8081`).** Point it
  at `:8081` and you bypass every policy in this workshop.
- **`type: openIdConnect`** selects the OIDC flow. STAC Browser also supports `apiKey` and
  HTTP Basic; OIDC is the one that works with a real identity provider.
- **`oidcConfig`** is passed through to
  [`oidc-client-ts`](https://authts.github.io/oidc-client-ts/interfaces/UserManagerSettings.html).
  `client_id` defaults to `stac-browser` anyway; we set it explicitly because it is the value
  the identity provider sees. `automaticSilentRenew` is off because our mock server issues no
  refresh tokens — leaving it on produces a failed renewal every few minutes.
- **The redirect URI is not configurable.** STAC Browser always uses
  `<its own origin>/auth`, so `http://localhost:8080/auth`. Your identity provider must
  allow that exact URL. (The mock server accepts any redirect URI; a real one will not.)

<div class="alert alert-block alert-warning">
<b>Gotcha:</b> OIDC in STAC Browser only works when <code>historyMode</code> is
<code>history</code>. That is the default, so leave it alone — but if you switch to hash
routing to simplify static hosting, login silently stops working.
</div>

## 8.3 Set up something worth looking at

Reusing chapter 7's policy: collections named `private-<owner>-*` are visible only to a JWT
whose `username` claim matches. We create one public collection and one private collection each
for alice and bob, then go look at them in the browser.

Unlike chapter 7, we leave this data in place until the end of the chapter.

In [ ]:
alice = auth_headers(get_mock_oidc_token("alice", claims={"username": "alice"}))
bob = auth_headers(get_mock_oidc_token("bob", claims={"username": "bob"}))

public_id = "public-demo"
alice_id = "private-alice-notebook"
bob_id = "private-bob-notebook"


def collection(cid, title, description):
    return {
        "id": cid,
        "type": "Collection",
        "title": title,
        "stac_version": "1.0.0",
        "description": description,
        "license": "CC-BY-4.0",
        "extent": {
            "spatial": {"bbox": [[-10, -10, 10, 10]]},
            "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
        },
        "links": [],
    }


for cid, headers, title, description in [
    (public_id, alice, "Public Demo", "Visible to everyone, signed in or not."),
    (alice_id, alice, "Alice's Private Data", "Only visible to alice."),
    (bob_id, bob, "Bob's Private Data", "Only visible to bob."),
]:
    response = httpx.put(
        f"{stac_api_endpoint}/collections/{cid}",
        headers=headers,
        json=collection(cid, title, description),
        timeout=10,
    )
    if response.status_code == 404:
        response = httpx.post(
            f"{stac_api_endpoint}/collections",
            headers=headers,
            json=collection(cid, title, description),
            timeout=10,
        )
    print(f"{cid} -> {response.status_code}")
    assert response.status_code in (200, 201), response.text

Confirm the policy is live before opening the UI — if this assertion fails, the browser will
just show you the same thing every time and the demo falls flat.

In [ ]:
def visible(headers):
    response = httpx.get(
        f"{stac_api_endpoint}/collections", headers=headers, timeout=10
    )
    response.raise_for_status()
    ids = {c["id"] for c in response.json()["collections"]}
    return ids & {public_id, alice_id, bob_id}


print(f"anonymous sees: {sorted(visible({}))}")
print(f"alice     sees: {sorted(visible(alice))}")
print(f"bob       sees: {sorted(visible(bob))}")

assert visible({}) == {public_id}, "row-level filtering is not active -- see chapter 7"
assert visible(alice) == {public_id, alice_id}
assert visible(bob) == {public_id, bob_id}

## 8.4 Walkthrough: watch the catalog change

This part is hands-on in your browser. Run the cell below for a live frame, or open
<http://localhost:8080> in a separate tab (a real tab is easier — the OIDC redirect
navigates away and back).

**Step 1 — anonymous.** Open <http://localhost:8080>. You should see `public-demo` and the
collections from earlier chapters. Neither private collection is listed. Not greyed out, not
"access denied" — simply absent, because pgSTAC never returned them.

**Step 2 — log in as alice.** Click **Login** (top right). STAC Browser redirects to the
mock identity server at `:8085`, which shows a small login form. Enter:

- **Username:** `alice`
- **Claims:** `{"username": "alice"}`

Submit. You land back on `http://localhost:8080/auth`, which hands the code to STAC Browser,
which exchanges it for a token and re-fetches the catalog.

**Step 3 — look again.** `private-alice-notebook` is now in the list. Same URL, same
endpoint, different rows — the `username` claim in your token changed what pgSTAC returned.
`private-bob-notebook` is still missing.

**Step 4 — be bob.** Log out, log back in as `bob` with claims `{"username": "bob"}`. The two
private collections swap places.

**Step 5 — try to cheat.** While logged in as bob, navigate directly to
<http://localhost:8080/collections/private-alice-notebook>. You get a not-found error, not a
permission error. Guessing the id gains nothing.

In [ ]:
from IPython.display import IFrame

IFrame("http://localhost:8080", width="100%", height=600)

<div class="alert alert-block alert-warning">
<b>Two mock-server rough edges.</b> Tokens expire after 15 minutes and there are no refresh
tokens, so a long session will quietly drop back to the anonymous view — log in again. And
the mock server publishes no <code>end_session_endpoint</code>, so <b>Logout</b> may raise an
error; switching users is most reliable in a private/incognito window, or by clearing site
data for <code>localhost:8080</code>. Both are artifacts of the mock provider, not of
stac-auth-proxy or STAC Browser.
</div>

## 8.5 What the browser is actually sending

Nothing exotic. After login, STAC Browser attaches the access token to every catalog
request as a normal bearer header — the same header chapters 6 and 7 built by hand:

```
Authorization: Bearer <access_token>
```

You can prove the equivalence: decode the claims from a token issued the same way STAC
Browser gets one, and confirm the `username` claim the filter keys on is present.

In [ ]:
import base64


def claims_of(token):
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


token = get_mock_oidc_token("alice", claims={"username": "alice"})
decoded = claims_of(token)

print(json.dumps({k: decoded[k] for k in ("sub", "scope", "username")}, indent=2))
assert decoded["username"] == "alice"

Which is worth stating plainly: **STAC Browser has no authorization logic in it.** It
obtains a token and forwards it. Every decision about what is visible happens in the proxy
and the database. Swap the browser for QGIS, `pystac-client`, or a notebook and the policy
holds — that is the point of enforcing it server-side rather than in each client.

## 8.6 Clean up

Remove the demo collections. Each owner deletes their own, because the chapter 7 filter
applies to deletes too.

In [ ]:
for cid, headers in [(alice_id, alice), (bob_id, bob), (public_id, alice)]:
    response = httpx.delete(
        f"{stac_api_endpoint}/collections/{cid}", headers=headers, timeout=10
    )
    print(f"DELETE {cid} -> {response.status_code}")
    assert response.status_code in (200, 204, 404), response.text

print("\nCleaned up.")

## 8.7 Takeaways

- The **Authentication Extension** lets an API advertise how to authenticate; the proxy adds
  it by default, and STAC Browser's `authConfig` uses the same scheme-object shape.
- Point the browser at the **proxy**, never the upstream API — the policy lives in the proxy.
- STAC Browser's redirect URI is fixed at `<origin>/auth`; register it with your identity
  provider.
- OIDC requires `historyMode: history`.
- Clients hold **no** authorization logic. They carry a token; the server decides. That is
  what makes the same policy hold across every STAC client your users bring.

## Where to go next

You now have the full lifecycle running locally: metadata in, catalog queryable, writes
gated by OIDC, rows filtered per user, and a UI that respects all of it. Moving this to a
real deployment mostly means swapping pieces out:

- Replace the mock server with a real IdP (Auth0, Keycloak, Cognito, Entra ID). The proxy
  only needs its discovery URL.
- Replace `TenantFilter` with your own policy — or delegate to
  [Open Policy Agent](https://www.openpolicyagent.org/) via the built-in `Opa` filter if you
  already run one.
- Set `ALLOWED_JWT_AUDIENCES` so tokens minted for other services are rejected.
- Switch the row-level policy to **default-deny** (see §7.3.2).
- Deploy with [eoapi-cdk](https://github.com/developmentseed/eoapi-cdk) or
  [eoapi-k8s](https://github.com/developmentseed/eoapi-k8s), putting the proxy in front of
  the STAC API.

<div class="alert alert-block alert-info">
<b>One honest caveat.</b> titiler-pgstac and tipg talk to the database directly and do not
pass through the proxy, so the policies in this workshop cover the <i>STAC API only</i>. A
user who cannot see a collection in STAC can still request tiles for its assets if they know
the ids. Securing the raster and vector services is a separate exercise.
</div>